# Laya ontology fine-tuning — Apple Silicon MPS / CPU

Adapted from the official Laya fine-tuning examples at commit `80a1da1fa5448ea12163f4d977935625a0aea670`.
Open this notebook from the repository root or from `training/`; ROOT resolves to the repository root. For Kaggle, upload the extracted package as a dataset; keep outputs in `/kaggle/working`. Requires internet to install packages and fetch model weights.

Data: run `./build_all.sh` first. `DATASETS` selects folders under `data/train/` (`all` by default); `training/load_data.py` loads `data/train/<name>/{train,calibration}.jsonl` and `data/test/<name>/test.jsonl` and drops any train/calibration row that overlaps any test row or a protected Bio-ML benchmark query. The Bio-ML 2026 comparison set lives in `data/test/bioml2026/benchmark/` and is scored separately with `evaluation/bioml_rank.py`.

Training and hardware execution were not run during preparation. Syntax, split validation and importer tests were checked. No automatic publishing.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json
ROOT = Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()  # repository root
assert (ROOT / "training" / "load_data.py").exists(), "run from the repository root or training/"
DATASETS = "all"  # or e.g. "bioml2026,oaei_anatomy,ner_bionlp"
os.environ["USE_TF"] = "0"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements-training.txt")])


## Run on a Mac
Use a native Apple Silicon Python/PyTorch environment for MPS. `auto` selects MPS when available, otherwise CPU. Explicit `mps` fails clearly if unavailable. CUDA AMP is disabled; computation uses float32. MPS fallback is enabled before torch imports in the child process. Unsupported operations may run on CPU or still require choosing CPU explicitly. If memory is tight, try head-only training with `--freeze-encoder` or reduce sequence length after verifying the token-budget check.


In [ ]:
OUTPUT = ROOT / "models" / "LayaLink"
DEVICE = "auto"
cmd = [sys.executable, str(ROOT / "training" / "train_layalink_mps.py"),
       "--data-root", str(ROOT), "--datasets", DATASETS, "--device", DEVICE,
       "--out", str(OUTPUT), "--epochs", "4", "--micro-batch", "1", "--grad-accum", "32"]
subprocess.check_call(cmd + ["--dry-run"])
subprocess.check_call(cmd)


## Inspect held-out results and load the checkpoint

In [ ]:
print((OUTPUT / "evaluation.json").read_text())
import laya
import torch
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
agent = laya.Agent(str(OUTPUT), device=device)
sys.path.insert(0, str(ROOT / "training"))
from load_data import load_splits
example = load_splits(ROOT, DATASETS)["test"][0]
print(json.dumps(agent.predict(example["state"], example["questions"])["answers"], indent=2))
